# Proyecto

Primero cargamos los datos y las librerias para usar

In [ ]:
install.packages("ggplot2")
install.packages("dplyr")
install.packages("lubridate")
install.packages("car")
install.packages("nortest")
install.packages("corrplot")
install.packages("bestNormalize")
install.packages("MASS")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(ggplot2)
library(dplyr)
library(lubridate)
library(ggplot2)
library(nortest)
library(corrplot)
library(car)
library(bestNormalize)
library(MASS)

In [ ]:
df <- read.csv(file='data/hour.csv', head=TRUE, sep=',')

df[0:5,]

### Revisamos datos nulos

In [ ]:
anyNA(df)

### Por comodidad y formato voy a tener la variable de la fecha en una sola columna.

## Revisando valores nulos [Notas del creador]

Sucede que R toma la informacion horaria del pais en donde se configuro el kernel, que corresponde a Chile en este caso; y si este país realiza cambios de hora, estos cambios generan problemas en la lectura de los datos, como en el 20 de agosto de 2011 a la medianoche, Chile adelantó sus relojes, esto provocó que en Chile la fecha-hora `2011-08-21 00:00:00` tecnicamente no exista, interpretandose como NaN, esta misma situación sucede en multiples otras fechas.

**solucion:**
Se usa `tz="UTC"`, que corresponde a la zona de tiempo utilizada en Washintong D.C., Estados Unidos, de esta forma, no existe conflicto entre las fechas-horas presentes en el dataset.

In [ ]:
df$dteday <- paste( #combinar columnas
    df$dteday,
    df$hr,
    sep = " "
)

df$dteday <- as.POSIXct(x = df$dteday, format = '%Y-%m-%d %H', tz="UTC") #transformarlos a variables tipo time

df[c("yr", "mnth", "hr", "cnt")] <- NULL #elimino variables que no sirven por ahora

df[0:5,]

In [ ]:
colSums(is.na(df)) # cuantos nan hay en cada columna
which(is.na(df$dteday)) # en que filas estan 

### graficamos para visualizar como se comportan las variables a predecir a tráves del tiempo

In [ ]:
df_diario <- df %>%
    group_by(dteday = floor_date(dteday, unit = "day")) %>%
    summarise(
        casual = mean(casual, na.rm = TRUE),
        registered = mean(registered, na.rm = TRUE),
        weekday = last(weekday)
    ) # ahora me doy cuenta de que pude hacer groupby con weekday nomas... xd

df_diario[0:5,]

In [ ]:
boxplot(data=df_diario, casual ~ weekday, col='green')

boxplot(data=df_diario, registered ~ weekday, col='blue')

In [ ]:
df_mensual <- df %>%
    filter(!is.na(dteday)) %>%
    group_by(dteday = floor_date(dteday, unit = "month")) %>%
    summarise(
        casual = mean(casual, na.rm = TRUE),
        registered = mean(registered, na.rm = TRUE)
    )

df_mensual[0:5,]


In [ ]:
ggplot(data=df_mensual, aes(x = dteday)) + 

    geom_line(aes(y = casual, color='casuales')) + 
    geom_point(aes(y = casual, color='casuales'), size = 2, shape=18)+

    geom_line(aes(y = registered, color='registrados')) +
    geom_point(aes(y = registered, color='registrados'), size = 2, shape = 16)+

    scale_color_manual(
        values = c(
            'casuales' = 'green',
            'registrados' = 'blue'
        )
    ) +

    labs(
        title='Comparacion de casuales y registrados',
        subtitle= 'soy un subtitulo :D',
        x='mes',
        y='cantidad promedio',
        color='variables'
    )

Por ahora, se puede decir que en el caso de los **usuarios registrados**, los días en los que hay un **ligero incremento en el promedio de alquiler de bicicletas son entre lunes y viernes.** Por otro lado, los **usuarios casuales** observan un **incremento en el uso de bicicletas durante el fin de semana**, específicamente los sábados y domingos. Finalmente, es relevante mencionar que tanto los usuarios casuales como los registrados sufren una **reducción en el alquiler de bicicletas en los meses que se acercan al fin de año**, que en Washinton D.C. corresponden a los meses invernales; además, ambos tipos de usuarios muestran un aumento a lo largo del tiempo, siendo el crecimiento de los registrados mucho más evidente.

Esto nos lleva a concluir que los datos **carecen de estacionalidad**, por lo que las **variables temporales serán muy relevantes** para los modelos de regresión.

Ahora, al examinar el histograma para observar la distribución de las variables de respuesta, podemos verificar si hay una normalidad en los datos. Además, se analizara para determinar si **es adecuado emplear una regresión de Poisson** tal como fue sugerido por el profesor.

In [ ]:
imprimir_histograma <- function(columna) {

    if (columna == "casual") {
        color <- "green"
    } else if (columna == "registered") {
        color <- "blue"
    } else {
        color <- "white"
    }

    hist(
        df[[columna]],
        main = paste("Histograma de", columna),
        xlab = "valores",
        ylab = "frecuencia",
        col = color,
        breaks = "Sturges", # calcula un break ideal mediante 1+log(n)
        freq = FALSE
    )

    lines(density(df[[columna]]), lwd = 5, col = "red")
}

In [ ]:
imprimir_histograma("registered")

In [ ]:
imprimir_histograma("casual")

In [ ]:
# Test de normalidad para muestras grandes, si fueran menos de 5000 se utiliza shapiro.test()
ad.test(df$registered) 
ad.test(df$casual)

Como se observa en los histogramas, las dos variables de respuesta no presentan una distribución que se asemeje a la normal. Según el test de normalidad de Anderson-Darling, ambas variables obtuvieron un valor p muy bajo, lo que nos lleva a rechazar $h_0$ y a concluir que **las variables de respuesta no son de una distribución normal**. Por otra parte, ambos histogramas muestran que la **variable respuesta se parece mucho mas a una distribucion Poisson**.

In [ ]:
summary(df$casual)
print("")
summary(df$registered)

In [ ]:
mean(df$casual)
var(df$casual)

mean(df$registered)
var(df$registered)

Los resultados mostraron que las dos variables presentan una varianza considerablemente mayor a sus medias, lo que va en contra de la idea de que sigan una distribución de Poisson, ya que en tal caso se debería cumplir que $esperanza \approx varianza$. Por lo tanto, por el momento, **no se justifica el uso de una regresión de Poisson**.

A continuación, se realizará un **análisis de las relaciones entre las variables**, para determinar si habrá **problemas de colinealidad** en los modelos y para identificar cuáles son las **mejores opciones como variables predictoras**.

In [ ]:
df_numeric <- df[, sapply(df, FUN=is.numeric)]

df_numeric$dteday <- as.numeric(df$dteday) # practicamente es lo mismo que instant

# conseguir mas info temporal
df_numeric$year <- as.numeric(year(df$dteday))
df_numeric$month <- as.numeric(month(df$dteday))
df_numeric$day <- as.numeric(day(df$dteday))
df_numeric$hour <- as.numeric(hour(df$dteday))

matriz_cor <- cor(df_numeric, use='complete.obs')

corrplot.mixed(corr = matriz_cor, lower = 'pie', upper = 'number', tl.pos = 'lt')

In [ ]:
cor_casual <- sort(
    abs(
        matriz_cor['casual', colnames(matriz_cor) != "casual" & colnames(matriz_cor) != "registered"]
    ), 
    decreasing = TRUE
)
cor_regis <- sort(
    abs(
        matriz_cor['registered', colnames(matriz_cor) != "casual" & colnames(matriz_cor) != "registered"]
    ), 
    decreasing = TRUE
)

In [ ]:
for (i in seq_along(cor_casual)) {
    print(paste(i, "- casual:", names(cor_casual)[i], "=", cor_casual[i]))
}

In [ ]:
for (i in seq_along(cor_casual)) {
    print(paste(i, "- registered:", names(cor_regis)[i], "=", cor_regis[i]))
}

Tanto las variables de temperatura `temp` y temperatura normalizada `atemp`, como el mes `month` y temporada `season` muestran, como era de esperarse, una correlación muy alta. Para obtener una mejor relación con las variables de respuesta, **se optará por usar la variable de temperatura y temporada**; evitando así el riesgo de introducir problemas de colinealidad en el modelo. Además, dejando a un lado la situación mencionada, **las variables son bastante independientes entre sí**, con correlaciones que no superan el umbral de 0.5. Esto nos permite experimentar con todas las variables en nuestro **modelo sin tener que enfrentarnos a problemas de colinealidad**, al menos en este momento en el contexto lineal.

De cualquier forma, para el análisis y para verificar la colinealidad y determinar si es necesario aplicar una regresión específica, se llevará a cabo un modelo de tipo forward.

In [ ]:
df <- df_numeric %>% mutate(
    across(
      .cols = c(year, month, day, hour, workingday, holiday, weekday, season, weathersit),
      .fns = as.factor
    )
  )

In [ ]:
boxplot(data=df, casual ~ hour)
boxplot(data=df, registered ~ hour)

In [ ]:
modelo <- lm(formula = casual ~ 1, data = df)

modelo_forward <- step(
    object = modelo,
    direction = "forward",
    scope = ~ temp + atemp + hum + hour + workingday + weathersit + year + season + windspeed + month
)

In [ ]:
vif(modelo_forward)

Hemos verificado recientemente que, utilizando el modelo forward, hay un **problema de VIF elevado para variables con más correlación**, ya que es superior a 10, por lo que para el modelo preliminar se llevará a cabo lo que se había sugerido anteriormente.

Ahora, dado que los **datos son muy parecidos a un poisson**, se propondrán tres transformaciones que pueden resolver el alto problema de varianza que presentan los datos. En primer lugar, sería una **transformación logarítmica** $Y_t^* = log(Y_t + 1)$ y una **transformación cuadrada** $Y_t^* = \sqrt{Y_t}$, ya que no tenemos valores negativos. Por ultimo una transformacion por medio de la **transformación de Yeo Johnson**

In [ ]:
df$casual_log <- log1p(df$casual)
df$registered_log <- log1p(df$registered)

df$casual_sqrt <- sqrt(df$casual)
df$registered_sqrt <- sqrt(df$registered)

In [ ]:
print(paste("media: ", mean(df$casual_log), "- varianza: ", var(df$casual_log)))
print(paste("media: ", mean(df$registered_log), "- varianza: ", var(df$registered_log)))

In [ ]:
imprimir_histograma('casual_log')
imprimir_histograma('registered_log')

In [ ]:
print(paste("media: ", mean(df$casual_sqrt), "- varianza: ", var(df$casual_sqrt)))
print(paste("media: ", mean(df$registered_sqrt), "- varianza: ", var(df$registered_sqrt)))

In [ ]:
imprimir_histograma('casual_sqrt')
imprimir_histograma('registered_sqrt')

In [ ]:
yeo_result_casual <- powerTransform(df$casual, family = 'yjPower')
summary(yeo_result_casual)
confint(yeo_result_casual)

In [ ]:
yeo_result_regis <- powerTransform(df$registered, family = 'yjPower')
summary(yeo_result_regis)
confint(yeo_result_regis)

In [ ]:
lambda_casual <- yeo_result_casual$lambda
lambda_regis <- yeo_result_regis$lambda

casual.t <- yeojohnson(df$casual, standardize = F)
regis.t <- yeojohnson(df$registered, standardize = F)

df$casual.t <- casual.t$x.t
df$regis.t <- regis.t$x.t

df$casual.t <- car::yjPower(df$casual, lambda_casual)
df$regis.t <- car::yjPower(df$registered, lambda_regis)

In [ ]:
print(paste("media: ", mean(df$casual.t), "- varianza: ", var(df$casual.t)))
print(paste("media: ", mean(df$regis.t), "- varianza: ", var(df$regis.t)))

In [ ]:
imprimir_histograma('casual.t')
imprimir_histograma('regis.t')

## Modelos

c ~ temp + hour + workingday + weathersit + year + season + windspeed + weekday + holiday
r ~ hour + temp + hum + year + season + workingday + weathersit + windspeed + holiday + weekday

In [ ]:
fecha_corte <- as.POSIXct("2012-10-01")

df_train <- df[df$dteday < fecha_corte, ]
df_test  <- df[df$dteday >= fecha_corte, ]

In [ ]:
modelo_c_lin <- lm(
    data=df_train,
    formula = casual ~ temp + hour + workingday + weathersit + year + season + windspeed + weekday + holiday
)
summary(modelo_c_lin)

In [ ]:
modelo_c_log <- lm(
    data=df_train,
    formula = casual_log ~ temp + hour + workingday + weathersit + year + season + windspeed + weekday + holiday
)
summary(modelo_c_log)

In [ ]:
modelo_c_sqrt <- lm(
    data=df_train,
    formula = casual_sqrt ~ temp + hour + workingday + weathersit + year + season + windspeed + weekday + holiday
)
summary(modelo_c_sqrt)

In [ ]:
modelo_c_ys <- lm(
    data=df_train,
    formula = casual.t ~ temp + hour + workingday + weathersit + year + season + windspeed + weekday + holiday
)
summary(modelo_c_ys)

In [ ]:
modelo_r_lin <- lm(
    data=df_train,
    formula = registered ~ hour + temp + hum + year + season + workingday + weathersit + windspeed + holiday + weekday
)
summary(modelo_r_lin)

In [ ]:
modelo_r_log <- lm(
    data=df_train,
    formula = registered_log ~ hour + temp + hum + year + season + workingday + weathersit + windspeed + holiday + weekday
)
summary(modelo_r_log)

In [ ]:
modelo_r_sqrt <- lm(
    data=df_train,
    formula = registered_sqrt ~ hour + temp + hum + year + season + workingday + weathersit + windspeed + holiday + weekday
)
summary(modelo_r_sqrt)

In [ ]:
modelo_r_ys <- lm(
    data=df_train,
    formula = regis.t ~ hour + temp + hum + year + season + workingday + weathersit + windspeed + holiday + weekday
)
summary(modelo_r_ys)

Tanto para `casual` como para `registered`, el **modelo que utiliza la transformación logarítmica presenta el mejor desempeño.** En ambos casos, este modelo obtiene un RSE cercano a 0.6, indicando una menor dispersión de los residuos. Además, presenta los mayores valores de $R^2$ y el ajustado, junto con el mayor estadístico F. Por lo tanto, entre las transformaciones evaluadas, la que parece ser mas adecuada es la transformación logarítmica.

Ahora para confirmar esto ultimo, se vera que tal son las predicciones

In [ ]:
pred_c_lin <- predict(modelo_c_lin, newdata=df_test)

# volvemos a escala original
pred_c_log <- expm1(predict(modelo_c_log, newdata=df_test)) 

pred_c_sqrt <- (predict(modelo_c_sqrt, newdata=df_test))^2

pred_c_ys <- predict(modelo_c_ys, newdata = df_test)
pred_c_ys <- (lambda_casual * pred_c_ys + 1)^(1 / lambda_casual) - 1

In [ ]:
data.frame(
  Modelo = c("Lineal", "Log", "Raíz", "Yeo-Johnson"),
  RMSE = c(
    sqrt(mean((df_test$casual - pred_c_lin)^2)),
    sqrt(mean((df_test$casual - pred_c_log)^2)),
    sqrt(mean((df_test$casual - pred_c_sqrt)^2)),
    sqrt(mean((df_test$casual - pred_c_ys)^2))
  ),
  MAE = c(
    mean(abs(df_test$casual - pred_c_lin)),
    mean(abs(df_test$casual - pred_c_log)),
    mean(abs(df_test$casual - pred_c_sqrt)),
    mean(abs(df_test$casual - pred_c_ys))
  )
)

In [ ]:
pred_r_lin <- predict(modelo_r_lin, newdata=df_test)

# volvemos a escala original
pred_r_log <- expm1(predict(modelo_r_log, newdata=df_test)) 

pred_r_sqrt <- (predict(modelo_r_sqrt, newdata=df_test))^2

pred_r_ys <- predict(modelo_r_ys, newdata = df_test)
pred_r_ys <- (lambda_regis * pred_r_ys + 1)^(1 / lambda_regis) - 1

In [ ]:
data.frame(
  Modelo = c("Lineal", "Log", "Raíz", "Yeo-Johnson"),
  RMSE = c(
    sqrt(mean((df_test$registered - pred_r_lin)^2)),
    sqrt(mean((df_test$registered - pred_r_log)^2)),
    sqrt(mean((df_test$registered - pred_r_sqrt)^2)),
    sqrt(mean((df_test$registered - pred_r_ys)^2))
  ),
  MAE = c(
    mean(abs(df_test$registered - pred_r_lin)),
    mean(abs(df_test$registered - pred_r_log)),
    mean(abs(df_test$registered - pred_r_sqrt)),
    mean(abs(df_test$registered - pred_r_ys))
  )
)

A partir de los resultados obtenidos para las variables `casual` y `registered`, se observa que el modelo **logarítmico presenta un buen desempeño en ambos casos**, especialmente al comparar los valores de RMSE y MAE. Por esta razón, por ahora se seleccionará el modelo logarítmico para ambas variables. Sin embargo, el modelo para `registered` todavía **presenta un error relativamente alto**, por lo que posteriormente se evaluará la posibilidad de utilizar otro enfoque, como un **modelo de Poisson**, que podría ser más adecuado para este tipo de datos de conteo.

En el modelo para `casual`, los coeficientes indican cómo las distintas variables afectan al logaritmo de la cantidad de usuarios casuales, manteniendo las demás variables constantes. Se observa que la `temperatura` tiene un efecto positivo importante, mientras que una mayor `velocidad del viento` se relaciona con una disminución en los usuarios. La `hora` también presenta diferencias importantes: durante la madrugada los coeficientes son negativos, mientras que durante el día se vuelven positivos, alcanzando sus mayores valores alrededor de las 17:00 horas. Los días laborales presentan una menor cantidad de usuarios casuales que los días no laborales, y las condiciones climáticas desfavorables también reducen la cantidad esperada de usuarios. Por otro lado, en 2012 se observa un aumento respecto a 2011, y las estaciones distintas del invierno presentan efectos positivos. Finalmente, el modelo presenta un **R² de 0.8154**, indicando que explica aproximadamente el 81.5% de la variabilidad del logaritmo de `casual`.

En el modelo para `registered`, los coeficientes representan el efecto de las distintas variables sobre el logaritmo de la cantidad de usuarios registrados, manteniendo las demás variables constantes. La `hora` presenta un efecto importante, con valores negativos durante la madrugada y positivos durante gran parte del día, alcanzando los mayores valores alrededor de las 17:00 y 18:00 horas. La `temperatura` tiene un efecto positivo, mientras que una mayor `humedad` y velocidad del viento se relacionan con una disminución en la cantidad de usuarios registrados. En 2012 se observa un aumento respecto a 2011, y las estaciones presentan efectos positivos respecto al invierno. Los días laborales también presentan una mayor cantidad de usuarios registrados que los días no laborales. Por otro lado, las condiciones climáticas desfavorables tienden a disminuir la cantidad de usuarios, especialmente en `weathersit3`. Finalmente, el modelo presenta un **R² de 0.8155**, por lo que explica aproximadamente el 81.5% de la variabilidad del logaritmo de `registered`, siendo estadísticamente significativo en su conjunto.

In [ ]:


modelo_poisson <- glm(casual ~ temp + weekday + hum + month + season + windspeed, data = df, family = poisson(link = "log"))

modelo_negbin <- glm.nb(casual ~ temp + weekday + hum + month + season + windspeed, data = df)
                  
cat("AIC Modelo Poisson: ", AIC(modelo_poisson), "\n")
cat("AIC Modelo Binomial Negativa: ", AIC(modelo_negbin), "\n\n")

dispersión_poisson <- summary(modelo_poisson)$deviance / summary(modelo_poisson)$df.residual
cat("Parámetro de sobredispersión estimado (Poisson): ", dispersión_poisson, "\n")
cat("(Valores significativamente mayores a 1 confirman la presencia de sobredispersión)")
summary(modelo_negbin)

Al comparar mediante el uso de el AIC de ambos modelos para la estimación de alquileres de usuarios casual obtuvimos los siguientes resultados:
1. Respecto a la Sobredispersión:
La regresión de Poisson asume de que, teóricamente, la media y la varianza son idénticas λ, sin embargo, luego de un test obtuvimos sobredispersión estimado de 20.21. Al ser considerablemente superior a 1, esto invalida el uso de la distribución de Poisson, ya que subestima demasiado los errores estándar de los coeficientes, inflando de sobremanera la falsa significancia de variables que no nos sirven, es decir, producto de esto le da valor a variables que no lo merecen.
2. Comparativa de Ajuste (AIC):
   - AIC Modelo Poisson: 425,370.9
   - AIC Modelo Binomial Negativa: 142,627.5
Luego de calcular el puntaje AIC de ambos modelos y revisar sus resultados es facil darnos cuennta de que el Modelo Binomial Negativo es considerablemente superior al Modelo Poisson. Esto confirma de que la Binomial Negativa logra describir de mejor manera el comportamiento de nuestro dataset, por lo que lo usaremos.
3. Variables importantes respecto al modelo:
Temperatura ("temp"): Es la variable predictora de mayor impacto sobre el alquiler casual. Tiene una relación directamente proporcional con la demanda.
Humedad ("hum"): Afecta de manera inversamente proporcional y en gran medida al alquiler casual, osea que el clima húmedo o las lluvias reducen el alquiler. Esta variable por obvias razones es tan importante como la temperatura